# Geocode Idealista Barcelona Addresses

This notebook extracts address-like text from `../data/idealista_barcelona_sale_urls.csv`, identifies listings with a street number, and geocodes only those usable addresses. In this file, the address-like text is stored in `address_search`; if a future export contains a true `description` column, the notebook will use that first.

## Plan

1. Load the Idealista URL/listing file.
2. Use the best available address text column: `description`, then `address_search`, then `description_search`.
3. Remove the property-type prefix before `in`, e.g. `Flat / apartment in Calle de Pau Claris, 76, ...` becomes `Calle de Pau Claris, 76, ...`.
4. Flag rows that contain a street number. Rows with only a neighborhood/street name are kept but not sent for exact geocoding.
5. Geocode unique full-address candidates with OpenStreetMap Nominatim, using a local CSV cache and a one-second delay.
6. Merge lat/lon back to all properties and export both row-level results and a summary.

In [1]:
import re
import time
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

d:\Users\Eric\Desktop\BSE\Master Thesis\scrape-homes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
INPUT_CSV = Path("../data/idealista_barcelona_sale_urls.csv")
OUTPUT_CSV = Path("../data/idealista_barcelona_sale_urls_geocoded.csv")
SUMMARY_CSV = Path("../data/idealista_barcelona_sale_urls_geocode_summary.csv")
CACHE_CSV = Path("../data/idealista_geocode_cache.csv")

# Nominatim requires a descriptive User-Agent. Replace the email/contact with yours if you run a large batch.
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
USER_AGENT = "scrape-idealista-barcelona-geocoder/1.0 (local research notebook)"
REQUEST_DELAY_SECONDS = 1.1
MAX_ADDRESSES_TO_GEOCODE = None  # set to a small integer, e.g. 25, for a test run

df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} property rows from {INPUT_CSV}")
df.head()

Loaded 10,619 property rows from ..\data\idealista_barcelona_sale_urls.csv


,propertyCode,url,address_search,price_search,details_search,description_search,source_page,scraped_at,seed_url,seed_label
0,109696793,https://www.idealista.com/en/inmueble/109696793/,"Flat / apartment in Calle d'Aragó, La Dreta de l'Eixample, Barcelona","1,199,000 €",3 bed. 164 m² 1st floor exterior with lift | 3 bed. | 164 m² | 1st floor exterior with lift,FOR SALE: Amazing and bright renovated modernist apartment with terrace in the Eixample. This exceptional property F...,https://www.idealista.com/en/venta-viviendas/barcelona-barcelona/,2026-04-17T15:35:12+00:00,NaN,NaN
1,110509586,https://www.idealista.com/en/inmueble/110509586/,"Flat / apartment in Calle de Tamarit, Sant Antoni, Barcelona","545,000 €",2 bed. 80 m² 4th floor exterior with lift | 2 bed. | 80 m² | 4th floor exterior with lift,We present this completely renovated apartment of great quality the apartment consists of two bedrooms two bathrooms...,https://www.idealista.com/en/venta-viviendas/barcelona-barcelona/,2026-04-17T15:35:12+00:00,NaN,NaN
2,110630744,https://www.idealista.com/en/inmueble/110630744/,"Flat / apartment in Calle del Consell de Cent, La Dreta de l'Eixample, Barcelona","1,390,000 €",3 bed. 139 m² 3rd floor exterior with lift | 3 bed. | 139 m² | 3rd floor exterior with lift,Excellent renovated apartment with original features FOR SALE in the Eixample. This flat FOR SALE is located in a st...,https://www.idealista.com/en/venta-viviendas/barcelona-barcelona/,2026-04-17T15:35:12+00:00,NaN,NaN
3,110063350,https://www.idealista.com/en/inmueble/110063350/,"Flat / apartment in Calle de Girona, La Dreta de l'Eixample, Barcelona","2,200,000 €",3 bed. 271 m² 1st floor exterior with lift | 3 bed. | 271 m² | 1st floor exterior with lift,Stunning refurbished modernist apartment with a large terrace in Eixample. This magnificent property is located in o...,https://www.idealista.com/en/venta-viviendas/barcelona-barcelona/,2026-04-17T15:35:12+00:00,NaN,NaN
4,110182478,https://www.idealista.com/en/inmueble/110182478/,"Flat / apartment in Calle de Mallorca, La Dreta de l'Eixample, Barcelona","2,195,000 €",5 bed. 221 m² 5th floor exterior with lift | 5 bed. | 221 m² | 5th floor exterior with lift,Outstanding brand new apartment in a stately building next to Paseo de Gracia. This wonderful flat has been complete...,https://www.idealista.com/en/venta-viviendas/barcelona-barcelona/,2026-04-17T15:35:12+00:00,NaN,NaN


## Extract Address Candidates

In [3]:
SOURCE_COLUMN_PRIORITY = ["description", "address_search", "description_search"]
source_col = next((col for col in SOURCE_COLUMN_PRIORITY if col in df.columns), None)
if source_col is None:
    raise ValueError(f"None of the expected source columns exist: {SOURCE_COLUMN_PRIORITY}")

print(f"Using `{source_col}` as the source column for address extraction.")

Using `address_search` as the source column for address extraction.


In [7]:
STREET_WORD_RE = re.compile(
    r"\b("
    r"calle|carrer|avenida|avinguda|av\.?|paseo|passeig|rambla|ronda|plaza|plaça|pasaje|passatge|"
    r"travessera|carretera|via|gran via|cam[ií]|baixada|torrent|riera|jard[ií]|moll|muelle|"
    r"pla|portal|pujades|sender|camino|cami"
    r")\b",
    flags=re.IGNORECASE,
)
STREET_NUMBER_RE = re.compile(r"(?<!\d)\d{1,4}(?:\s*[-/]\s*\d{1,4})?(?:\s*[A-Za-z])?(?!\d)")
BAD_PLACEHOLDER_RE = re.compile(r"ask the advertiser|photos", flags=re.IGNORECASE)


def clean_spaces(value):
    return re.sub(r"\s+", " ", str(value)).strip()


def extract_address_candidate(value):
    if pd.isna(value):
        return ""
    text = clean_spaces(value)
    if not text or BAD_PLACEHOLDER_RE.search(text):
        return ""
    # Idealista labels usually look like: "Flat / apartment in Calle de Pau Claris, 76, ...".
    match = re.search(r"\bin\s+(.+)$", text, flags=re.IGNORECASE)
    if match:
        text = match.group(1).strip()
    return text.strip(" ,")


def has_street_word(value):
    return bool(value and STREET_WORD_RE.search(value))


def has_street_number(value):
    return bool(value and STREET_NUMBER_RE.search(value))


def clean_unicode(text):
    """Normalize unicode and replace common Catalan accents for better Nominatim matching."""
    import unicodedata
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text


def standardize_to_catalan(text):
    """Convert Spanish street names to Catalan for better OSM matching in Barcelona."""
    if not text:
        return text
    
    # Mapping of Spanish → Catalan street type names (case-insensitive)
    spanish_to_catalan = {
        r"\bcalle\b": "carrer",
        r"\bavenida\b": "avinguda", 
        r"\bav\.": "avinguda",  # Av. → Avinguda
        r"\bpaseo\b": "passeig",
        r"\bpasaje\b": "passatge",
        r"\btraversía\b": "travessera",
        r"\bcarretera\b": "carretera",  # Same in both
        r"\bvía\b": "via",  # Same in both
        r"\bplaza\b": "plaça",
        r"\bronda\b": "ronda",  # Same in both
        r"\brambla\b": "rambla",  # Same in both
        r"\bcamino\b": "camí",
        r"\bcamí\b": "camí",
        r"\bjardín\b": "jardí",
        r"\bmoll\b": "moll",  # Same in both
    }
    
    result = text
    for spanish_pattern, catalan_word in spanish_to_catalan.items():
        result = re.sub(spanish_pattern, catalan_word, result, flags=re.IGNORECASE)
    
    return result


def extract_street_address_only(address):
    """Extract just the street and number, before neighborhood/district info."""
    if not address:
        return address
    # Match: [Street content], [Number] - everything before the first comma is the street
    # This avoids including neighborhood/district names
    match = re.search(r"^([^,]+),\s*(\d{1,4}(?:\s*[-/]\s*\d{1,4})?(?:\s*[A-Za-z])?)\b", address, re.IGNORECASE)
    if match:
        street_part = match.group(1).strip()
        number_part = match.group(2).strip()
        return f"{street_part}, {number_part}"
    return address


def build_geocode_query(address):
    """Build primary geocoding query (street + number)."""
    if not address:
        return ""
    # Extract just the street and number (before district info)
    street_only = extract_street_address_only(address)
    query = clean_spaces(street_only)
    # Standardize Spanish to Catalan street names for Barcelona OSM matching
    query = standardize_to_catalan(query)
    # Clean unicode for better Nominatim matching
    query = clean_unicode(query)
    # Only add Barcelona if it's truly missing
    if not re.search(r"\bbarcelona\b", query, flags=re.IGNORECASE):
        query = f"{query}, Barcelona"
    if not re.search(r"\b(spain|españa|espanya)\b", query, flags=re.IGNORECASE):
        query = f"{query}, Spain"
    return query


def build_fallback_query(address):
    """Build fallback query using just the neighborhood/area name (for addresses without street numbers)."""
    if not address:
        return ""
    # Look for neighborhood name (usually after the first comma)
    # Pattern: street names, [NEIGHBORHOOD]
    parts = address.split(",")
    if len(parts) > 1:
        for part in parts[1:]:
            cleaned = part.strip()
            # Skip if it's just digits (likely a postal code or district number)
            if len(cleaned) > 3 and not cleaned.isdigit() and re.search(r"[a-zA-Z]", cleaned):
                # Clean unicode
                cleaned = clean_unicode(cleaned)
                query = f"{cleaned}, Barcelona"
                if not re.search(r"\b(spain|españa|espanya)\b", query, flags=re.IGNORECASE):
                    query = f"{query}, Spain"
                return query
    # If no neighborhood found, try geocoding just the street name (before any number or neighborhood)
    street_match = re.match(r"^([^,\d]+)", address)
    if street_match:
        street_name = street_match.group(1).strip()
        if street_name:
            street_name = standardize_to_catalan(street_name)
            street_name = clean_unicode(street_name)
            query = f"{street_name}, Barcelona, Spain"
            return query
    return ""


def parse_price_eur(value):
    if pd.isna(value):
        return pd.NA
    match = re.search(r"\d[\d.,]*", str(value))
    if not match:
        return pd.NA
    number_text = match.group(0).replace(".", "").replace(",", "")
    try:
        return int(number_text)
    except ValueError:
        return pd.NA


# Build work dataframe with both primary and fallback geocoding queries
work = df.copy()
if "price_search" in work.columns:
    work["price_eur"] = work["price_search"].map(parse_price_eur).astype("Int64")
else:
    work["price_eur"] = pd.Series(pd.NA, index=work.index, dtype="Int64")

work["address_source_column"] = source_col
work["address_source_text"] = work[source_col]
work["address_candidate"] = work["address_source_text"].map(extract_address_candidate)
work["has_address_candidate"] = work["address_candidate"].ne("")
work["has_street_word"] = work["address_candidate"].map(has_street_word)
work["has_street_number"] = work["address_candidate"].map(has_street_number)

# PRIMARY QUERY: Exact address with street number (high confidence)
work["geocode_eligible_primary"] = work["has_address_candidate"] & work["has_street_number"]
work["geocode_query_primary"] = work["address_candidate"].where(work["geocode_eligible_primary"], "").map(build_geocode_query)

# FALLBACK QUERY: Neighborhood/area geocoding for addresses without street numbers (lower confidence)
work["geocode_eligible_fallback"] = work["has_address_candidate"] & ~work["has_street_number"]
work["geocode_query_fallback"] = work["address_candidate"].where(work["geocode_eligible_fallback"], "").map(build_fallback_query)

# For backwards compatibility, mark ALL addresses with queries as geocode_eligible
work["geocode_eligible"] = work["geocode_query_primary"].ne("") | work["geocode_query_fallback"].ne("")
# Use primary query if available, otherwise fallback
work["geocode_query"] = work["geocode_query_primary"].fillna("") + work["geocode_query_fallback"].fillna("")
work["geocode_query"] = work["geocode_query"].replace("", pd.NA)

work[["propertyCode", source_col, "address_candidate", "has_street_number", "geocode_eligible_primary", "geocode_eligible_fallback", "geocode_query"]].head(20)

,propertyCode,address_search,address_candidate,has_street_number,geocode_eligible_primary,geocode_eligible_fallback,geocode_query
0,109696793,"Flat / apartment in Calle d'Aragó, La Dreta de l'Eixample, Barcelona","Calle d'Aragó, La Dreta de l'Eixample, Barcelona",False,False,True,"La Dreta de l'Eixample, Barcelona, Spain"
1,110509586,"Flat / apartment in Calle de Tamarit, Sant Antoni, Barcelona","Calle de Tamarit, Sant Antoni, Barcelona",False,False,True,"Sant Antoni, Barcelona, Spain"
2,110630744,"Flat / apartment in Calle del Consell de Cent, La Dreta de l'Eixample, Barcelona","Calle del Consell de Cent, La Dreta de l'Eixample, Barcelona",False,False,True,"La Dreta de l'Eixample, Barcelona, Spain"
3,110063350,"Flat / apartment in Calle de Girona, La Dreta de l'Eixample, Barcelona","Calle de Girona, La Dreta de l'Eixample, Barcelona",False,False,True,"La Dreta de l'Eixample, Barcelona, Spain"
4,110182478,"Flat / apartment in Calle de Mallorca, La Dreta de l'Eixample, Barcelona","Calle de Mallorca, La Dreta de l'Eixample, Barcelona",False,False,True,"La Dreta de l'Eixample, Barcelona, Spain"
5,106144229,"Flat / apartment in La Dreta de l'Eixample, Barcelona","La Dreta de l'Eixample, Barcelona",False,False,True,"Barcelona, Barcelona, Spain"
6,109488303,"Flat / apartment in Calle de Girona, La Dreta de l'Eixample, Barcelona","Calle de Girona, La Dreta de l'Eixample, Barcelona",False,False,True,"La Dreta de l'Eixample, Barcelona, Spain"
7,110055339,"Flat / apartment in Calle del Comte d'Urgell, L'Antiga Esquerra de l'Eixample, Barcelona","Calle del Comte d'Urgell, L'Antiga Esquerra de l'Eixample, Barcelona",False,False,True,"L'Antiga Esquerra de l'Eixample, Barcelona, Spain"
8,111026222,"Flat / apartment in Calle de Roger de Llúria, La Dreta de l'Eixample, Barcelona","Calle de Roger de Llúria, La Dreta de l'Eixample, Barcelona",False,False,True,"La Dreta de l'Eixample, Barcelona, Spain"
9,110823853,"Flat / apartment in Calle de Pau Claris, 76, La Dreta de l'Eixample, Barcelona","Calle de Pau Claris, 76, La Dreta de l'Eixample, Barcelona",True,True,False,"carrer de Pau Claris, 76, Barcelona, Spain"


## Coverage Summary Before Geocoding

In [8]:
summary = pd.DataFrame(
    {
        "metric": [
            "properties_pulled",
            "nonblank_source_text",
            "address_candidates_extracted",
            "candidates_with_street_word",
            "candidates_with_street_number (primary)",
            "candidates_without_street_number (fallback)",
            "geocode_eligible_primary_addresses",
            "geocode_eligible_fallback_addresses",
            "unique_primary_geocode_queries",
            "unique_fallback_geocode_queries",
        ],
        "count": [
            len(work),
            work["address_source_text"].notna().sum(),
            work["has_address_candidate"].sum(),
            work["has_street_word"].sum(),
            work["has_street_number"].sum(),
            work["geocode_eligible_fallback"].sum(),
            work["geocode_eligible_primary"].sum(),
            work["geocode_eligible_fallback"].sum(),
            work.loc[work["geocode_eligible_primary"], "geocode_query_primary"].nunique(),
            work.loc[work["geocode_eligible_fallback"], "geocode_query_fallback"].nunique(),
        ],
    }
)
summary["share_of_properties"] = summary["count"] / len(work)
summary

,metric,count,share_of_properties
0,properties_pulled,10619,1.000000
1,nonblank_source_text,10619,1.000000
2,address_candidates_extracted,10584,0.996704
3,candidates_with_street_word,7443,0.700913
4,candidates_with_street_number (primary),1174,0.110557
5,candidates_without_street_number (fallback),9410,0.886147
6,geocode_eligible_primary_addresses,1174,0.110557
7,geocode_eligible_fallback_addresses,9410,0.886147
8,unique_primary_geocode_queries,879,0.082776
9,unique_fallback_geocode_queries,70,0.006592


In [6]:
print(f"Properties pulled: {len(work):,}")
print(f"Rows with address candidates: {work['has_address_candidate'].sum():,}")
print(f"Rows with street-number addresses (primary): {work['geocode_eligible_primary'].sum():,}")
print(f"Rows with area-only addresses (fallback): {work['geocode_eligible_fallback'].sum():,}")
print(f"Unique primary geocode queries: {work.loc[work['geocode_eligible_primary'], 'geocode_query_primary'].nunique():,}")
print(f"Unique fallback geocode queries: {work.loc[work['geocode_eligible_fallback'], 'geocode_query_fallback'].nunique():,}")

Properties pulled: 10,619
Rows with address candidates: 10,584
Rows with street-number addresses (primary): 1,174
Rows with area-only addresses (fallback): 9,410
Unique primary geocode queries: 879
Unique fallback geocode queries: 70


Rows without a street number are usually not exact enough for property-level geocoding. Keep them in the output for auditing, but do not send them as exact addresses.

In [15]:
work.loc[
    work["geocode_eligible_fallback"],
    ["propertyCode", "address_candidate", "has_street_word", "has_street_number", "url"],
].head(25)

,propertyCode,address_candidate,has_street_word,has_street_number,url
0,109696793,"Calle d'Aragó, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/109696793/
1,110509586,"Calle de Tamarit, Sant Antoni, Barcelona",True,False,https://www.idealista.com/en/inmueble/110509586/
2,110630744,"Calle del Consell de Cent, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/110630744/
3,110063350,"Calle de Girona, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/110063350/
4,110182478,"Calle de Mallorca, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/110182478/
5,106144229,"La Dreta de l'Eixample, Barcelona",False,False,https://www.idealista.com/en/inmueble/106144229/
6,109488303,"Calle de Girona, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/109488303/
7,110055339,"Calle del Comte d'Urgell, L'Antiga Esquerra de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/110055339/
8,111026222,"Calle de Roger de Llúria, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/111026222/
10,108818765,"Calle de la Diputació, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/108818765/


## Geocode Unique Eligible Addresses

This uses OpenStreetMap Nominatim. For a large or repeated project, consider a paid geocoding provider or a local Nominatim instance. The local cache keeps prior results and makes reruns cheap.

In [9]:
def load_cache(path):
    if path.exists():
        cache = pd.read_csv(path)
        if "query" in cache.columns:
            cache = cache.drop_duplicates("query", keep="last")
        return cache
    return pd.DataFrame(columns=["query", "lat", "lon", "display_name", "importance", "osm_type", "osm_id", "geocode_status", "geocoded_at"])


def save_cache(cache, path):
    cache.drop_duplicates("query", keep="last").to_csv(path, index=False)


def geocode_one(query):
    params = {
        "q": query,
        "format": "jsonv2",
        "limit": 1,
        "addressdetails": 1,
        "countrycodes": "es",
    }
    response = requests.get(NOMINATIM_URL, params=params, headers={"User-Agent": USER_AGENT}, timeout=30)
    response.raise_for_status()
    results = response.json()
    if not results:
        return {
            "query": query,
            "lat": pd.NA,
            "lon": pd.NA,
            "display_name": "",
            "importance": pd.NA,
            "osm_type": "",
            "osm_id": pd.NA,
            "geocode_status": "not_found",
            "geocoded_at": pd.Timestamp.utcnow().isoformat(),
        }
    best = results[0]
    return {
        "query": query,
        "lat": float(best.get("lat")) if best.get("lat") is not None else pd.NA,
        "lon": float(best.get("lon")) if best.get("lon") is not None else pd.NA,
        "display_name": best.get("display_name", ""),
        "importance": best.get("importance", pd.NA),
        "osm_type": best.get("osm_type", ""),
        "osm_id": best.get("osm_id", pd.NA),
        "geocode_status": "found",
        "geocoded_at": pd.Timestamp.utcnow().isoformat(),
    }

In [10]:
cache = load_cache(CACHE_CSV)
cached_queries = set(cache["query"].dropna()) if len(cache) else set()

# Collect ALL queries (primary + fallback)
primary_queries = sorted(work.loc[work["geocode_eligible_primary"], "geocode_query_primary"].dropna().unique())
fallback_queries = sorted(work.loc[work["geocode_eligible_fallback"], "geocode_query_fallback"].dropna().unique())

# Remove fallback queries that are already in primary or cached
all_queries = list(dict.fromkeys(primary_queries + fallback_queries))  # Remove duplicates while preserving order
all_queries = [q for q in all_queries if q]

queries_to_fetch = [query for query in all_queries if query and query not in cached_queries]
if MAX_ADDRESSES_TO_GEOCODE is not None:
    queries_to_fetch = queries_to_fetch[:MAX_ADDRESSES_TO_GEOCODE]

print(f"Total queries (primary + fallback): {len(all_queries):,}")
print(f"Primary queries: {len(primary_queries):,}")
print(f"Fallback queries: {len(fallback_queries):,}")
print(f"Already cached: {len(cached_queries & set(all_queries)):,}")
print(f"Queries to fetch this run: {len(queries_to_fetch):,}")

Total queries (primary + fallback): 949
Primary queries: 879
Fallback queries: 70
Already cached: 0
Queries to fetch this run: 949


In [11]:
new_rows = []
for query in tqdm(queries_to_fetch):
    try:
        new_rows.append(geocode_one(query))
    except Exception as exc:
        new_rows.append(
            {
                "query": query,
                "lat": pd.NA,
                "lon": pd.NA,
                "display_name": "",
                "importance": pd.NA,
                "osm_type": "",
                "osm_id": pd.NA,
                "geocode_status": f"error: {type(exc).__name__}: {exc}",
                "geocoded_at": pd.Timestamp.utcnow().isoformat(),
            }
        )
    time.sleep(REQUEST_DELAY_SECONDS)

if new_rows:
    cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
    save_cache(cache, CACHE_CSV)

cache = load_cache(CACHE_CSV)
cache.head()

  0%|          | 0/949 [00:00<?, ?it/s]C:\Users\eric\AppData\Local\Temp\ipykernel_17592\174511335.py:47: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "geocoded_at": pd.Timestamp.utcnow().isoformat(),
  1%|          | 9/949 [00:17<31:39,  2.02s/it]C:\Users\eric\AppData\Local\Temp\ipykernel_17592\174511335.py:35: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "geocoded_at": pd.Timestamp.utcnow().isoformat(),
100%|██████████| 949/949 [33:16<00:00,  2.10s/it]


,query,lat,lon,display_name,importance,osm_type,osm_id,geocode_status,geocoded_at
0,"Actor Morano, 2 -4, Barcelona, Spain",41.413553,2.105543,"Carrer de l'Actor Morano, Vallvidrera, Vallvidrera, el Tibidabo i les Planes, Sarrià - Sant Gervasi, Barcelona, Barc...",0.053418,way,3.850491e+08,found,2026-05-01T15:30:30.391847+00:00
1,"Amargos, 15, Barcelona, Spain",41.386430,2.174088,"15, Carrer d'Amargós, la Catedral, el Gòtic, Ciutat Vella, Barcelona, Barcelonès, Barcelona, Catalunya, 08002, España",0.000084,node,6.291115e+09,found,2026-05-01T15:30:31.994517+00:00
2,"Antic De Valencia, 99, Barcelona, Spain",41.403076,2.199409,"Camí Antic de València, el Poblenou, Sant Martí, Barcelona, Barcelonès, Barcelona, Catalunya, 08005, España",0.071949,way,6.858195e+08,found,2026-05-01T15:30:33.601569+00:00
3,"Av Republica Argentina, 94, Barcelona, Spain",41.265897,1.957296,"Avinguda República Argentina, Baixador, Castelldefels, Baix Llobregat, Barcelona, Catalunya, 08860, España",0.040061,way,6.733092e+08,found,2026-05-01T15:30:35.173520+00:00
4,"Avinguda Diagonal, 358, Barcelona, Spain",41.399871,2.171931,"358, Avinguda Diagonal, la Dreta de l'Eixample, l'Eixample, Barcelona, Barcelonès, Barcelona, Catalunya, 08013, España",0.000084,node,6.302917e+09,found,2026-05-01T15:30:38.444319+00:00


## Merge Geocodes and Export

In [12]:
geo_cols = ["query", "lat", "lon", "display_name", "importance", "osm_type", "osm_id", "geocode_status", "geocoded_at"]

# Merge on primary query first
geocoded = work.merge(
    cache[geo_cols], 
    left_on="geocode_query_primary", 
    right_on="query", 
    how="left",
    suffixes=("", "_primary")
).drop(columns=["query"], errors="ignore")

geocoded["geocode_match_type"] = "primary"

# For rows that didn't match on primary, try fallback query
unmatched_mask = geocoded["lat"].isna() & geocoded["geocode_query_fallback"].notna()
if unmatched_mask.any():
    fallback_merges = geocoded.loc[unmatched_mask].merge(
        cache[geo_cols],
        left_on="geocode_query_fallback",
        right_on="query",
        how="left",
        suffixes=("_x", "_fallback")
    ).drop(columns=["query"], errors="ignore")
    
    # Update unmatched rows with fallback results
    fallback_cols = ["lat", "lon", "display_name", "importance", "osm_type", "osm_id", "geocode_status", "geocoded_at"]
    for col in fallback_cols:
        if f"{col}_fallback" in fallback_merges.columns:
            geocoded.loc[unmatched_mask, col] = fallback_merges[f"{col}_fallback"].values
        elif col in fallback_merges.columns and f"{col}_x" not in fallback_merges.columns:
            geocoded.loc[unmatched_mask, col] = fallback_merges[col].values
    
    geocoded.loc[unmatched_mask, "geocode_match_type"] = "fallback"

geocoded["geocode_found"] = geocoded["lat"].notna() & geocoded["lon"].notna()

final_summary = pd.DataFrame(
    {
        "metric": [
            "properties_pulled",
            "rows_with_address_candidates",
            "rows_with_street_number_addresses (primary)",
            "rows_with_area_only_addresses (fallback)",
            "rows_eligible_for_geocoding",
            "rows_with_lat_lon",
            "rows_matched_by_primary_query",
            "rows_matched_by_fallback_query",
        ],
        "count": [
            len(geocoded),
            geocoded["has_address_candidate"].sum(),
            geocoded["has_street_number"].sum(),
            geocoded["geocode_eligible_fallback"].sum(),
            geocoded["geocode_eligible_primary"].sum() + geocoded["geocode_eligible_fallback"].sum(),
            geocoded["geocode_found"].sum(),
            (geocoded["geocode_match_type"] == "primary").sum(),
            (geocoded["geocode_match_type"] == "fallback").sum(),
        ],
    }
)
final_summary["share_of_properties"] = final_summary["count"] / len(geocoded)

geocoded.to_csv(OUTPUT_CSV, index=False)
final_summary.to_csv(SUMMARY_CSV, index=False)

print(f"Wrote row-level geocoded data to {OUTPUT_CSV}")
print(f"Wrote summary to {SUMMARY_CSV}")
final_summary

Wrote row-level geocoded data to ..\data\idealista_barcelona_sale_urls_geocoded.csv
Wrote summary to ..\data\idealista_barcelona_sale_urls_geocode_summary.csv


,metric,count,share_of_properties
0,properties_pulled,10619,1.000000
1,rows_with_address_candidates,10584,0.996704
2,rows_with_street_number_addresses (primary),1174,0.110557
3,rows_with_area_only_addresses (fallback),9410,0.886147
4,rows_eligible_for_geocoding,10584,0.996704
5,rows_with_lat_lon,10110,0.952067
6,rows_matched_by_primary_query,1072,0.100951
7,rows_matched_by_fallback_query,9547,0.899049


## Plot Geocoded Addresses

In [13]:
import plotly.express as px

plot_points = geocoded.loc[geocoded["geocode_found"]].copy()
plot_points["lat"] = pd.to_numeric(plot_points["lat"], errors="coerce")
plot_points["lon"] = pd.to_numeric(plot_points["lon"], errors="coerce")
plot_points["price_eur_float"] = pd.to_numeric(plot_points["price_eur"], errors="coerce")
plot_points = plot_points.dropna(subset=["lat", "lon"])

if plot_points.empty:
    print("No rows with lat/lon yet. Run the geocoding cells first, then rerun this cell.")
else:
    lon_pad = max((plot_points["lon"].max() - plot_points["lon"].min()) * 0.08, 0.01)
    lat_pad = max((plot_points["lat"].max() - plot_points["lat"].min()) * 0.08, 0.01)
    lon_range = [plot_points["lon"].min() - lon_pad, plot_points["lon"].max() + lon_pad]
    lat_range = [plot_points["lat"].min() - lat_pad, plot_points["lat"].max() + lat_pad]

    fig = px.scatter_geo(
        plot_points,
        lon="lon",
        lat="lat",
        color="price_eur_float",
        height=700,
        hover_name="address_candidate",
        hover_data={
            "propertyCode": True,
            "price_search": True,
            "price_eur_float": ":,.0f",
            "lat": ":.6f",
            "lon": ":.6f",
        },
        color_continuous_scale="Viridis",
        title="Geocoded Listings Colored by Price",
    )
    fig.update_traces(marker={"size": 7, "opacity": 0.75})
    fig.update_geos(
        projection_type="mercator",
        lonaxis_range=lon_range,
        lataxis_range=lat_range,
        showland=True,
        landcolor="#f3efe6",
        showocean=True,
        oceancolor="#d7edf7",
        showlakes=True,
        lakecolor="#d7edf7",
        showcountries=True,
        countrycolor="#aaaaaa",
        coastlinecolor="#777777",
        showframe=False,
    )
    fig.update_layout(template="plotly_white", margin={"r": 20, "t": 50, "l": 20, "b": 20})
    fig.show()

    raw_fig = px.scatter_geo(
        plot_points,
        lon="lon",
        lat="lat",
        height=700,
        hover_name="address_candidate",
        hover_data={
            "propertyCode": True,
            "price_search": True,
            "lat": ":.6f",
            "lon": ":.6f",
        },
        title="Raw Geocoded Listings: One Dot per Listing",
    )
    raw_fig.update_traces(marker={"size": 5, "opacity": 0.45, "color": "#2f6fbb"})
    raw_fig.update_geos(
        projection_type="mercator",
        lonaxis_range=lon_range,
        lataxis_range=lat_range,
        showland=True,
        landcolor="#f3efe6",
        showocean=True,
        oceancolor="#d7edf7",
        showlakes=True,
        lakecolor="#d7edf7",
        showcountries=True,
        countrycolor="#aaaaaa",
        coastlinecolor="#777777",
        showframe=False,
    )
    raw_fig.update_layout(template="plotly_white", margin={"r": 20, "t": 50, "l": 20, "b": 20})
    raw_fig.show()

In [14]:
geocoded.loc[
    geocoded["geocode_eligible"],
    ["propertyCode", "price_search", "price_eur", "address_candidate", "geocode_query", "lat", "lon", "geocode_status", "display_name", "url"],
].head(25)

,propertyCode,price_search,price_eur,address_candidate,geocode_query,lat,lon,geocode_status,display_name,url
0,109696793,"1,199,000 €",1199000,"Calle d'Aragó, La Dreta de l'Eixample, Barcelona","La Dreta de l'Eixample, Barcelona, Spain",41.396840,2.168752,found,"la Dreta de l'Eixample, l'Eixample, Barcelona, Barcelonès, Barcelona, Catalunya, España",https://www.idealista.com/en/inmueble/109696793/
1,110509586,"545,000 €",545000,"Calle de Tamarit, Sant Antoni, Barcelona","Sant Antoni, Barcelona, Spain",41.378412,2.161768,found,"Sant Antoni, l'Eixample, Barcelona, Barcelonès, Barcelona, Catalunya, 08015, España",https://www.idealista.com/en/inmueble/110509586/
2,110630744,"1,390,000 €",1390000,"Calle del Consell de Cent, La Dreta de l'Eixample, Barcelona","La Dreta de l'Eixample, Barcelona, Spain",41.396840,2.168752,found,"la Dreta de l'Eixample, l'Eixample, Barcelona, Barcelonès, Barcelona, Catalunya, España",https://www.idealista.com/en/inmueble/110630744/
3,110063350,"2,200,000 €",2200000,"Calle de Girona, La Dreta de l'Eixample, Barcelona","La Dreta de l'Eixample, Barcelona, Spain",41.396840,2.168752,found,"la Dreta de l'Eixample, l'Eixample, Barcelona, Barcelonès, Barcelona, Catalunya, España",https://www.idealista.com/en/inmueble/110063350/
4,110182478,"2,195,000 €",2195000,"Calle de Mallorca, La Dreta de l'Eixample, Barcelona","La Dreta de l'Eixample, Barcelona, Spain",41.396840,2.168752,found,"la Dreta de l'Eixample, l'Eixample, Barcelona, Barcelonès, Barcelona, Catalunya, España",https://www.idealista.com/en/inmueble/110182478/
5,106144229,"1,460,000 €",1460000,"La Dreta de l'Eixample, Barcelona","Barcelona, Barcelona, Spain",41.382580,2.177073,found,"Barcelona, Barcelonès, Barcelona, Catalunya, España",https://www.idealista.com/en/inmueble/106144229/
6,109488303,"1,790,000 €",1790000,"Calle de Girona, La Dreta de l'Eixample, Barcelona","La Dreta de l'Eixample, Barcelona, Spain",41.396840,2.168752,found,"la Dreta de l'Eixample, l'Eixample, Barcelona, Barcelonès, Barcelona, Catalunya, España",https://www.idealista.com/en/inmueble/109488303/
7,110055339,"960,000 €",960000,"Calle del Comte d'Urgell, L'Antiga Esquerra de l'Eixample, Barcelona","L'Antiga Esquerra de l'Eixample, Barcelona, Spain",41.390000,2.155000,found,"l'Antiga Esquerra de l'Eixample, l'Eixample, Barcelona, Barcelonès, Barcelona, Catalunya, España",https://www.idealista.com/en/inmueble/110055339/
8,111026222,"2,700,000 €",2700000,"Calle de Roger de Llúria, La Dreta de l'Eixample, Barcelona","La Dreta de l'Eixample, Barcelona, Spain",41.396840,2.168752,found,"la Dreta de l'Eixample, l'Eixample, Barcelona, Barcelonès, Barcelona, Catalunya, España",https://www.idealista.com/en/inmueble/111026222/
9,110823853,"2,150,000 €",2150000,"Calle de Pau Claris, 76, La Dreta de l'Eixample, Barcelona","carrer de Pau Claris, 76, Barcelona, Spain",41.389449,2.171632,found,"76, Carrer de Pau Claris, la Dreta de l'Eixample, l'Eixample, Barcelona, Barcelonès, Barcelona, Catalunya, 08010, Es...",https://www.idealista.com/en/inmueble/110823853/


In [18]:
data = pd.read_csv('../data/idealista_barcelona_sale_urls_geocoded.csv')
data[data['geocode_match_type']=='primary'].shape

(1072, 33)